# Metadata

In [ ]:
# load metadata
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent.parent
filepath = ROOT / "data" / "brset"  / "metadata.csv"

df = pd.read_csv(filepath)
df.head()

In [ ]:
df = df.copy()
to_drop = ['patient_age', 'illuminaton', 'diabetic_retinopathy', 'dr_icdr', 'dr_sdrg']

df.columns = map(str.lower, df.columns)

df['age_approx'] = df['patient_age']
df['sex'] = df['patient_sex'].map({1: 'male', 2: 'female'}).fillna('unknown')
df['illumination'] = df['illuminaton']
df["anatom_site_general"] = df["exam_eye"].map({1: "right_eye", 2: "left_eye"}).fillna("unknown")
df["target"] = df["diabetic_retinopathy"].astype(int)

df.drop(to_drop, axis=1, inplace=True)

In [ ]:
# create train/test sets
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5,
                                shuffle=True,
                                random_state=42,
)

train_idx, test_idx = next(splitter.split(df,
                                          y=df["target"],
                                          groups=df["patient_id"],
                                          )
)

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

In [ ]:
overlap = set(train_df["patient_id"]) & set(test_df["patient_id"])
assert len(overlap) == 0, f"Patient leakage detected: {len(overlap)} patients overlap"

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))

print("\nTrain target distribution:")
print(train_df["target"].value_counts(normalize=True))

print("\nTest target distribution:")
print(test_df["target"].value_counts(normalize=True))

In [ ]:
out_dir = ROOT / "data" / 'brset' / 'processed'
out_dir.mkdir(parents=True, exist_ok=True)

train_path = out_dir / "train-metadata.parquet"
test_path = out_dir / "test-metadata.parquet"

train_df.to_parquet(train_path, index=False, engine="pyarrow")
test_df.to_parquet(test_path, index=False, engine="pyarrow")

print(f"Saved train: {train_path}")
print(f"Saved test: {test_path}")

# Images

In [ ]:
from PIL import Image
img_path = ROOT / "data" / "brset" / "images" / 'img00042.jpg'
img = Image.open(img_path)
print(img.size)
display(img)

In [ ]:
from pathlib import Path
import io
import h5py
import numpy as np
from PIL import Image
from tqdm import tqdm


def image_to_jpeg_bytes(image_path, size=(256, 256), quality=95):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img = img.resize(size, Image.Resampling.LANCZOS)

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)

        return buffer.getvalue()


def create_hdf5_from_metadata(df, image_dir, out_filepath, id_col="image_id", size=(256, 256)):
  image_dir = Path(image_dir)
  out_filepath = Path(out_filepath)
  out_filepath.parent.mkdir(parents=True, exist_ok=True)

  with h5py.File(out_filepath, "w") as h5:
      h5.attrs["format"] = "jpeg_bytes"
      h5.attrs["image_size"] = str(size)
      h5.attrs["id_col"] = id_col

      for image_id in tqdm(df[id_col].astype(str), desc=f"Creating {out_filepath.name}"):
          image_path = image_dir / f"{image_id}.jpg"

          if not image_path.exists():
              raise FileNotFoundError(f"Missing image: {image_path}")

          jpeg_bytes = image_to_jpeg_bytes(image_path, size=size)
          h5.create_dataset(image_id, data=np.bytes_(jpeg_bytes))

  print(f"Created: {out_filepath}")

In [ ]:
# load csv
image_dir = ROOT / "data" / "brset" / "images"
processed_dir = ROOT / "data" / "brset" / "processed"

train_metadata_path = processed_dir / "train-metadata.parquet"
test_metadata_path = processed_dir / "test-metadata.parquet"

train_df = pd.read_parquet(train_metadata_path)
test_df = pd.read_parquet(test_metadata_path)

In [ ]:
create_hdf5_from_metadata(
    df=train_df,
    image_dir=image_dir,
    out_filepath=processed_dir / "train-image.hdf5",
    id_col="image_id",
    size=(512, 512),
)

create_hdf5_from_metadata(
    df=test_df,
    image_dir=image_dir,
    out_filepath=processed_dir / "test-image.hdf5",
    id_col="image_id",
    size=(512, 512),
)

In [ ]:
#sanity-check
image_id = "img00042"

with h5py.File(processed_dir / "train-image.hdf5", "r") as h5:
    file_bytes = h5[image_id][()].tobytes()

img = Image.open(io.BytesIO(file_bytes)).convert("RGB")
print(img.size)
display(img)